# Feature Engineering 

Análisis exploratorio para el diseño de features del modelo de forecast de `prod_pet` (petróleo, horizonte t+1).

**Reglas que sigue este notebook:**

- Todo el análisis se hace **solo sobre `train`** (periodo ≤ 2023-07-01, ADR-028). No se mira `val` ni `test` — ni siquiera para definir qué pozos entran.
- El **universo** es petrolero: pozos con al menos un mes de `prod_pet > 0`, calculado **dentro de train**.
- Las funciones y el diccionario de columnas viven en `ml/eda.py` (no se define lógica inline acá).
- Las fechas de corte salen de `ml/config.py`, idénticas a las del modelado.

## Índice
1. Carga del dataset crudo (train) y primeras filas
2. Diccionario de columnas — qué significa cada una

In [1]:
import sys
from pathlib import Path

# permite importar el paquete `ml` desde el notebook (raíz del repo en el path)
REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from ml.config import TRAIN_END
from ml import eda

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", None)  # no truncar descripciones largas

## 1. Carga del dataset crudo (train) y primeras filas

Cargamos las **40 columnas crudas** del export de producción, recortadas a train y al universo petrolero (ver `ml/eda.load_train_raw`).

In [2]:
train = eda.load_train_raw()

print(f"Filas (train, universo petrolero): {len(train):,}")
print(f"Pozos únicos:                      {train['idpozo'].nunique():,}")
print(f"Columnas crudas:                   {train.shape[1] - 1}  (+ 'periodo' derivada)")
print(f"Rango temporal:                    {train['periodo'].min().date()}  ->  {train['periodo'].max().date()}")
print(f"Corte de train (TRAIN_END):        {TRAIN_END.date()}")

Filas (train, universo petrolero): 224,166
Pozos únicos:                      3,018
Columnas crudas:                   40  (+ 'periodo' derivada)
Rango temporal:                    2006-01-01  ->  2023-07-01
Corte de train (TRAIN_END):        2023-07-01


**Primeras filas.** Con 40 columnas, la vista normal queda ilegible a lo ancho; la mostramos **transpuesta** (una columna por fila) para leer de un vistazo qué trae cada campo en los primeros registros.

In [3]:
train.head(5).T

,0,1,2,3,4
idempresa,PEL,PEL,PEL,PEL,PEL
anio,2006,2006,2006,2006,2006
mes,2,3,4,5,6
idpozo,3640,3640,3640,3640,3640
prod_pet,0.0,30.808,76.609,63.273,55.043
prod_gas,0.0,1.731,3.713,4.202,4.373
prod_agua,0.0,30.57,46.479,50.536,52.062
iny_agua,0.0,0.0,0.0,0.0,0.0
iny_gas,0.0,0.0,0.0,0.0,0.0
iny_co2,0.0,0.0,0.0,0.0,0.0


## 2. Diccionario de columnas — qué significa cada una

Descripción de las 40 columnas crudas, con su **rol** (identificador, temporal, medida, categórica, atributo del pozo, geografía o metadata) y su **unidad** cuando aplica. El target (`prod_pet`) queda marcado en su descripción.

Fuente: Secretaría de Energía (data.gob.ar) — producción mensual de pozos de gas y petróleo. Definición canónica del modelo dimensional en [`docs/data-model.md`](../docs/data-model.md).

In [4]:
eda.data_dictionary()

,rol,unidad,descripción
columna,,,
idpozo,identificador,—,ID único del pozo (clave natural del pozo).
idempresa,identificador,—,ID de la empresa operadora (clave natural de la operadora).
idusuario,metadata,—,ID del usuario que cargó/rectificó el registro.
idareapermisoconcesion,identificador,—,ID del área de permiso/concesión.
idareayacimiento,identificador,—,ID del yacimiento/área (clave natural del yacimiento).
anio,temporal,año,Año del registro de producción.
mes,temporal,1-12,Mes del registro de producción.
fechaingreso,metadata,fecha,Fecha de ingreso del registro al sistema de la Secretaría.
fecha_data,metadata,fecha,Fecha/versión del dato (usada para deduplicar rectificaciones).


Si querés ver el conteo por rol (cuántas columnas son medidas, categóricas, etc.):

In [5]:
eda.data_dictionary()["rol"].value_counts().rename("n_columnas").to_frame()

,n_columnas
rol,
categórica,15
medida,8
metadata,6
identificador,4
atributo_pozo,3
temporal,2
geografía,2


## 3. Nulls por columna cruda

Cuántos valores faltantes tiene cada columna en train, en % sobre las 224.166 filas. Ordenado de mayor a menor; la barra es ayuda visual del porcentaje.

> Nota: las medidas numéricas se castean con `errors="coerce"` al cargar, así que cualquier valor no numérico en origen también cuenta como null.

In [6]:
eda.null_report_styled(train)

,rol,dtype,n_nulls,pct_null
columna,,,,
vida_util,atributo_pozo,float64,"220,424",98.33%
observaciones,metadata,object,"211,690",94.43%
subclasificacion,categórica,object,778,0.35%
clasificacion,categórica,object,778,0.35%
tipoextraccion,categórica,object,369,0.16%
tipopozo,categórica,object,369,0.16%
tipoestado,categórica,object,369,0.16%
sub_tipo_recurso,categórica,object,308,0.14%
idempresa,identificador,object,0,0.00%


**Lectura.** El dataset crudo está **muy completo**: 32 de las 40 columnas no tienen ningún null en train. Solo 8 tienen faltantes, y casi todas en proporción despreciable:

- `vida_util` (**98%**) y `observaciones` (**94%**) → prácticamente vacías, **se descartan** como features.
- `subclasificacion`, `clasificacion` (~0,35%), `tipoextraccion`, `tipopozo`, `tipoestado`, `sub_tipo_recurso` (<0,2%) → faltantes mínimos; si alguna se usa como feature, alcanza con imputar una categoría `"DESCONOCIDO"`.

Ninguna **medida** de producción (`prod_pet`, `prod_gas`, etc.) tiene nulls, así que el target y los candidatos numéricos están completos.